The purpose of this notebook is to verify the correctness of the functions in `utils_hamiltonian.py`, and present simple examples of their usage

In [15]:
import numpy as np
from utils_hamiltonian import (
    split_pauli_operator,
    pauli_matrix_element_with_basis_state,
    remove_irrelevant_pauli_terms,
    taper_hamiltonian,
    CNOT_clifford_operator,
    seniority_solving_clifford_operator,
    append_seniority_solving_clifford_circuit,
    group_odds_and_evens,
    ungroup_odds_and_evens
)
from openfermion import (
    QubitOperator,
    get_sparse_operator,
    get_fermion_operator,
    MolecularData,
    jordan_wigner
)
from openfermionpyscf import run_pyscf
from utils_basic import (
    random_pauli_term, 
    random_bin_list, 
    state_from_bin_list,
    compute_product_of_unitaries,
    random_pauli_hamiltonian,
    apply_unitary_product
)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from numpy.random import uniform
import random

## Pauli Operator Matrix Elements

Here, I verify the following:
- `split_pauli_operator` should cleave a Pauli operator into two Pauli operators correctly


In [ ]:
P = QubitOperator('X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9')
N = 5
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


P = QubitOperator('X0 Y5 Y6 Y7 Y8 Y9')
N = 3
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')

Here, I verify the following:

- <v|P|w> is calculated correctly for a bunch of P, v, w examples by comparing with numerical value

In [ ]:
for Nqubits in range(2, 12):
    for _ in range(40):
        _, P = random_pauli_term(Nqubits)
        v    = random_bin_list(Nqubits)
        w    = random_bin_list(Nqubits)

        matrix_element_numerical  = state_from_bin_list(v) @ get_sparse_operator(P, Nqubits) @ state_from_bin_list(w)
        matrix_element_analytical = pauli_matrix_element_with_basis_state(P, v, w)
        
        print(np.abs(matrix_element_numerical - matrix_element_analytical) < 1e-12, end='')
    print()

## Hamiltonian modification based on symmetries

First, I generate a simple non-trivial Hamiltonian to work with

In [ ]:
geo = [
    ['H', [0,0,0]],
    ['H', [0,0,1]],
    ['H', [0,0,2]],
    ['H', [0,0,3]]
]
mol = run_pyscf(
    MolecularData(geo, 'sto3g', 1, 0), 
    run_scf=True
)
H = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))

# play around with this example by modifying v and w
v = [0,1,1,0]
w = [1,0,0,1]

Hrel = remove_irrelevant_pauli_terms(H, v, w)

print('Hrel')
print(Hrel)
print()

Htapered = taper_hamiltonian(H, v, w)

print('Htapered')
print(Htapered)

print()
print(Htapered - taper_hamiltonian(Hrel, v, w))

The purpose of this code is to modify the Hamiltonian without changing the relevant matrix element and/or expectation values. So we should see if that works as desired.

Here, I see if, for states $|\Psi_s\rangle = |s\rangle |\psi_s\rangle$, and $|\Psi_t\rangle = |t\rangle |\psi_t\rangle$, we have

$$
\langle \Psi_s|H|\Psi_t\rangle = \langle \Psi_s|H_{\text{rel}}|\Psi_t\rangle = \langle \psi_s|H_\text{tapered}|\psi_t \rangle
$$

I will use the LiH Hamiltonian since it is larger

In [ ]:
geo = [
    ['H', [0,0,0]],
    ['Li', [0,0,1]]
]
mol     = run_pyscf(MolecularData(geo, 'sto3g', 1, 0), run_scf=True)
H       = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))
Nqubits = 12

for _ in range(20):
    taper_size = random.sample(range(2, 8), 1)[0]

    bin_s = random_bin_list(taper_size)
    bin_t = random_bin_list(taper_size)

    ket_s = state_from_bin_list(bin_s, taper_size)
    ket_t = state_from_bin_list(bin_t, taper_size)

    psi_s = uniform(-1, 1, 2 ** (Nqubits - taper_size))
    psi_t = uniform(-1, 1, 2 ** (Nqubits - taper_size))

    Psi_s = np.kron(ket_s, psi_s)
    Psi_t = np.kron(ket_t, psi_t)


    Hsparse         = get_sparse_operator(H, Nqubits)
    ev_full         = Psi_s @ Hsparse @ Psi_t

    Hrel_sparse     = get_sparse_operator(remove_irrelevant_pauli_terms(H, bin_s, bin_t), Nqubits)
    ev_rel          = Psi_s @ Hrel_sparse @ Psi_t

    Htapered_sparse = get_sparse_operator(taper_hamiltonian(H, bin_s, bin_t), Nqubits - taper_size)
    ev_tapered      = psi_s @ Htapered_sparse @ psi_t

    print(f'''
        full expectation    : {ev_full}
        rel expectation     : {ev_rel}
        tapered expectation : {ev_tapered}

        all equal           : {np.abs(ev_full-ev_rel)<1e-12 and np.abs(ev_full-ev_tapered)<1e-12 and np.abs(ev_rel-ev_tapered)<1e-12}
    ''')

## Solving Seniority Operators

First, I check that $ e^{-i\pi/4} U_\text{CNOT}(0,1) = \text{CNOT}(0,1)$ and $e^{-i \pi / 4} U_\text{CNOT}(1,0) = \text{CNOT}(1,0)$

In [ ]:
CNOT_01 = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,0,1],
    [0,0,1,0]
])

CNOT_10 = np.array([
    [1,0,0,0],
    [0,0,0,1],
    [0,0,1,0],
    [0,1,0,0]
])

CNOT_01_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(0,1)), 2).toarray()

CNOT_10_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(1,0)), 2).toarray()


print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_01_from_func, CNOT_01), end=''
)

print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_10_from_func, CNOT_10)
)

Now I check if the seniority solving operator and the seniority solving circuits are the same

In [ ]:
for Nqubits in [2,4,6,8,10,12]:
    Uof = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits).toarray()

    qc = QuantumCircuit(Nqubits)
    append_seniority_solving_clifford_circuit(qc, Nqubits)

    Uqk = Operator(qc).reverse_qargs().data

    print(
        np.allclose(np.exp(-1j * np.pi * (Nqubits // 2) / 4) * Uof, Uqk), end=''
    )

TrueTrueTrueTrueTrueTrue

Now, I check if $U z_{2i} z_{2i+1} U^T = z_{2i}$

In [18]:
Nqubits = 12
Ucliff  = seniority_solving_clifford_operator(Nqubits)

for i in range(Nqubits // 2):
    parity  = QubitOperator(f'Z{2*i} Z{2*i+1}')
    rotated = apply_unitary_product(parity, Ucliff)
    print(rotated == QubitOperator(f'Z{2*i}'), end='')

TrueTrueTrueTrueTrueTrue

## Full Tapering Procedure for Seniority

First: I check manually the reordering maps. First: applying both should produce the Hamiltonian I started with.

In [ ]:
for Nqubits in range(2, 12):
    for Nterms in range(Nqubits // 2, 5 * Nqubits):
        H       = random_pauli_hamiltonian(Nqubits, Nterms)
        Hrecov1 = ungroup_odds_and_evens(group_odds_and_evens(H, Nqubits), Nqubits)
        Hrecov2 = group_odds_and_evens(ungroup_odds_and_evens(H, Nqubits), Nqubits)

        print(H - Hrecov1, end='')
        print(H - Hrecov2, end='')
        print(Hrecov1 - Hrecov2, end='')
    print()

000000000000000000000000000
000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


Next, I check the reordering maps on some simple examples.

In [5]:
H          = QubitOperator('X0 Y1 X2 Y3 X4 Y5')
Hgrouped   = group_odds_and_evens(H, Nqubits)
Hungrouped = ungroup_odds_and_evens(Hgrouped, Nqubits)
Nqubits    = 6

print(f'''
    H          : {list(H.terms.keys())}
    Hgrouped   : {list(Hgrouped.terms.keys())}
    Hungrouped : {list(Hungrouped.terms.keys())}

    {H - Hungrouped == QubitOperator().zero()}
''')


    H          : [((0, 'X'), (1, 'Y'), (2, 'X'), (3, 'Y'), (4, 'X'), (5, 'Y'))]
    Hgrouped   : [((0, 'X'), (1, 'X'), (2, 'X'), (3, 'Y'), (4, 'Y'), (5, 'Y'))]
    Hungrouped : [((0, 'X'), (1, 'Y'), (2, 'X'), (3, 'Y'), (4, 'X'), (5, 'Y'))]

    True



Next, I check if, for a seniority conserving Hamiltonian, applying the Clifford transformation and regrouping produces zs on the first N/2 qubits

In [23]:
def duplicated_pauli_term_qubit_wise(term):
    """
    XYIZ -> XXYYZZ
    doesn't work with identities
    """
    newterm = []
    for op in term:
        newterm.append((2*op[0], op[1]))
        newterm.append((2*op[0]+1, op[1]))
    return newterm

def duplicate_hamiltonian_qubit_wise(H):
    newH = QubitOperator()
    for term, coef in H.terms.items():
        newH += coef * QubitOperator(duplicated_pauli_term_qubit_wise(term))
    return newH


Nqubits = 12

H = duplicate_hamiltonian_qubit_wise(
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Y1 X2 Y3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 X1 Z2 X3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 Y1 X2 X3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 X1 Z2 Y3 Z4 Z5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 X1 X2 Y3 X4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 Y1 Z2 Z3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Z1 X2 X3 Z4 Y5')
)

Ucliff = seniority_solving_clifford_operator(Nqubits)

Hrotated_temp           = apply_unitary_product(H, Ucliff)

Hrotated = QubitOperator()
for term, coef in Hrotated_temp.terms.items():
    Hrotated += np.round(coef, 4) * QubitOperator(term)

Hrotated_reordered = group_odds_and_evens(Hrotated, Nqubits)

print(H)

print()

print(Hrotated)

print()

print(Hrotated_reordered)

-1.5881 [X0 X1 X2 X3 Z4 Z5 Y6 Y7 Z8 Z9 Z10 Z11] +
1.3419 [X0 X1 Y2 Y3 X4 X5 Y6 Y7 X8 X9 X10 X11] +
-1.5614 [X0 X1 Z2 Z3 X4 X5 X6 X7 Z8 Z9 Y10 Y11] +
-0.4776 [Y0 Y1 X2 X3 Z4 Z5 X6 X7 Z8 Z9 Y10 Y11] +
-0.6995 [Y0 Y1 Y2 Y3 X4 X5 X6 X7 X8 X9 X10 X11] +
0.5475 [Z0 Z1 X2 X3 X4 X5 Y6 Y7 X8 X9 Y10 Y11] +
0.7721 [Z0 Z1 Y2 Y3 Z4 Z5 Z6 Z7 Z8 Z9 Y10 Y11]

-0.6995 [Z0 X1 Z2 X3 X5 X7 X9 X11] +
-0.4776 [Z0 X1 X3 Z4 X7 Z8 Z10 X11] +
0.7721 [Z0 Z2 X3 Z4 Z6 Z8 Z10 X11] +
0.5475 [Z0 X3 X5 Z6 X7 X9 Z10 X11] +
1.3419 [X1 Z2 X3 X5 Z6 X7 X9 X11] +
1.5614 [X1 Z2 X5 X7 Z8 Z10 X11] +
1.5881 [X1 X3 Z4 Z6 X7 Z8 Z10]

0.7721 [Z0 Z1 Z2 Z3 Z4 Z5 X7 X11] +
-0.6995 [Z0 Z1 X6 X7 X8 X9 X10 X11] +
-0.4776 [Z0 Z2 Z4 Z5 X6 X7 X9 X11] +
0.5475 [Z0 Z3 Z5 X7 X8 X9 X10 X11] +
1.3419 [Z1 Z3 X6 X7 X8 X9 X10 X11] +
1.5614 [Z1 Z4 Z5 X6 X8 X9 X11] +
1.5881 [Z2 Z3 Z4 Z5 X6 X7 X9]


It is instructive to see how it maps all seniority conserving operators actually.

In [26]:
XX = QubitOperator('X0 X1')
YY = QubitOperator('Y0 Y1')
ZZ = QubitOperator('Z0 Z1')
II = QubitOperator('')
Ucliff = seniority_solving_clifford_operator(Nqubits=2)

print(apply_unitary_product(XX, Ucliff))
print(apply_unitary_product(YY, Ucliff))
print(apply_unitary_product(ZZ, Ucliff))
print(apply_unitary_product(II, Ucliff))

0.9999999999999996 [X1]
-0.9999999999999996 [Z0 X1]
0.9999999999999996 [Z0]
0.9999999999999996 []
